# Generate an APPS solution with Codex, then grade it on Modal

This demo uses the official pip package [`openai-codex`](https://learn.chatgpt.com/docs/codex-sdk) with an existing **ChatGPT-backed Codex subscription login**. It starts with **1 standard-input/output question**, asks for a standalone Python implementation, and sends the generated source to the Modal evaluator introduced by the ground-truth demo.

Install this branch's CPU or GPU requirements in the `stego` environment. The Codex SDK includes its pinned CLI runtime. Sign in to Codex with ChatGPT beforehand; this notebook rejects API-key accounts and does not fall back to API billing. Subscription availability follows the current account/plan. “Max” is not a separate Codex credential type; the demonstrated route is [Sign in with ChatGPT](https://learn.chatgpt.com/docs/auth).

The kernel also needs `STEGO_ARTIFACTS_DIR` and normal Modal credentials (`MODAL_TOKEN_ID`/`MODAL_TOKEN_SECRET`, or an existing Modal login). The generation and grading cells are separate: generated source is saved before Modal is contacted. Grading uses billable Modal CPU resources, and only Modal executes the solution.

In [ ]:
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field

repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "ciphers/variable_naming_in_python_v2/data/codex_apps.py").is_file())
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
from ciphers.variable_naming_in_python_v2.data.apps import AppsConfig, AppsTestCases, load_apps
from ciphers.variable_naming_in_python_v2.data.codex_apps import CodexAppsConfig, build_stdio_prompt, generate_stdio_solution
from ciphers.variable_naming_in_python_v2.data.modal_apps import EVALUATOR_REVISION, ModalAppsConfig, evaluate_on_modal


class DemoConfig(BaseModel):
    model_config = ConfigDict(extra="forbid")
    num_problems: int = Field(default=1, ge=1)
    seed: int = 42


demo = DemoConfig()
apps_config = AppsConfig(min_tests=1)  # Native introductory stdio tasks have one supplied pair in this snapshot.
codex_config = CodexAppsConfig(model=None, timeout_s=180)  # None uses Codex's configured default model.
modal_config = ModalAppsConfig(case_timeout_s=4, solution_timeout_s=120, memory_mb=1024)
problems = load_apps(apps_config)
# Normalize JSON strings as well as mappings; large integer cases may remain serialized.
stdio_indices = [index for index, row in enumerate(problems) if AppsTestCases.from_dataset_value(row["input_output"]).fn_name is None]
stdio_problems = problems.select(stdio_indices)
sample = stdio_problems.shuffle(seed=demo.seed).select(range(min(demo.num_problems, len(stdio_problems))))
if not len(sample):
    raise ValueError("No standard-input/output APPS questions match these filters")
print(f"Selected {len(sample)} of {len(stdio_problems)} eligible stdin/stdout questions")

The prompt describes stdin/stdout behavior and preserves the full question, including its public examples. It contains **no supplied reference solutions or hidden test cases**. This demo selects APPS's native stdio tasks rather than changing the interface of function-call problems. Codex is asked for a JSON `code` field containing the entire program; local validation parses Python syntax without executing it.

The notebook explicitly uses `min_tests=1`: all eligible introductory stdio records in the pinned snapshot have one supplied input/output pair. That pair may contain multiple problem-level cases. The loader and ground-truth demo retain their default minimum of 10 pairs. Passing one supplied pair does not establish broader test coverage.

In [ ]:
for problem in sample:
    print(build_stdio_prompt(problem))
    print("=" * 80)

In [ ]:
generated = []
for problem in sample:
    print(f"Generating APPS {apps_config.split}/{problem['problem_id']} with Codex", flush=True)
    answer = await generate_stdio_solution(problem, codex_config)
    generated.append((problem, answer))
    print(answer.code)
    print("Generation saved:", Path(answer.artifact_dir) / "answer.json")

The next cell executes generated source remotely against **all supplied cases**. Modal gives each solution a fresh CPU sandbox with no user secrets and blocked network access. The official APPS evaluator retains its permissive output comparisons. Raw verdicts are `true`/`false`, `-1` (runtime error or per-case timeout), or `-2` (compilation/initialization failure); total process deadlines are reported separately. A failing generated program is a valid experiment outcome.

In [ ]:
records = []
for problem, answer in generated:
    cases = AppsTestCases.from_dataset_value(problem["input_output"])
    print(f"Grading APPS {apps_config.split}/{problem['problem_id']} on {problem['num_tests']} cases", flush=True)
    verdict = evaluate_on_modal(answer.code, cases, modal_config)
    records.append(
        {
            "problem_id": problem["problem_id"],
            "url": problem["url"],
            "generation": answer.model_dump(),
            "input_output": problem["input_output"],
            "verdict": verdict.model_dump(),
        }
    )
    print(f"{verdict.status}: {verdict.passed_tests}/{verdict.num_tests} cases passed")
    print("Raw APPS verdicts:", verdict.raw_results)
    if verdict.error or verdict.logs:
        print(verdict.error or "")
        print(verdict.logs)

In [ ]:
artifact_root = (repo_root / os.environ["STEGO_ARTIFACTS_DIR"]).resolve()
output_dir = artifact_root / "datasets/apps/codex-modal"
output_dir.mkdir(parents=True, exist_ok=True)
report_path = output_dir / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".json")
report = {
    "demo": demo.model_dump(),
    "apps_config": apps_config.model_dump(mode="json"),
    "codex_config": codex_config.model_dump(mode="json"),
    "modal_config": modal_config.model_dump(),
    "evaluator_revision": EVALUATOR_REVISION,
    "records": records,
}
report_path.write_text(json.dumps(report, indent=2))
print(f"Fully passing generated solutions: {sum(record['verdict']['status'] == 'passed' for record in records)}/{len(records)}")
print("Saved report:", report_path)